# EDA 1 - Audit 4 Cleaned Training Files

Notebook này chỉ dùng để đọc và audit 4 file clean trong `old/` trước khi viết `model_v3`.
Không train model, không tạo submission, không sửa dữ liệu gốc.


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

FILES = {
    'fixed_ffill': 'old/train_fixed_ffill.csv',
    'fixed_median': 'old/train_fixed_median.csv',
    'outlier_fixed_ffill': 'old/train_outlier_fixed_ffill.csv',
    'outlier_fixed_median': 'old/train_outlier_fixed_median.csv',
}

FLOAT_COLS = ['Quantity', 'UnitPrice', 'SalesAmount', 'Unit Cost', 'Cost Amount']
USECOLS = ['Date', 'ItemCode'] + FLOAT_COLS

print('Files:')
for name, path in FILES.items():
    p = Path(path)
    print(f'{name:24s}', p.exists(), f'{p.stat().st_size/1024/1024:.2f} MB' if p.exists() else '')


Files:
fixed_ffill              True 43.18 MB
fixed_median             True 43.18 MB
outlier_fixed_ffill      True 43.18 MB
outlier_fixed_median     True 43.17 MB


In [3]:
def load_clean_file(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=USECOLS, dtype={'ItemCode': 'category'}, parse_dates=['Date'])
    for col in FLOAT_COLS:
        df[col] = (
            df[col].astype(str)
            .str.replace(',', '.', regex=False)
            .astype('float32')
        )
    return df

print('[1] Loading datasets...')
datasets = {name: load_clean_file(path) for name, path in FILES.items()}
for name, df in datasets.items():
    print(name, df.shape, df['Date'].min().date(), '->', df['Date'].max().date(), 'skus=', df['ItemCode'].nunique())


[1] Loading datasets...
fixed_ffill (711980, 7) 2020-11-17 -> 2025-09-05 skus= 15972
fixed_median (711980, 7) 2020-11-17 -> 2025-09-05 skus= 15972
outlier_fixed_ffill (711980, 7) 2020-11-17 -> 2025-09-05 skus= 15972
outlier_fixed_median (711980, 7) 2020-11-17 -> 2025-09-05 skus= 15972


## 1. Schema / Shape Audit


In [4]:
schema_rows = []
for name, df in datasets.items():
    schema_rows.append({
        'dataset': name,
        'rows': len(df),
        'cols': df.shape[1],
        'sku_count': df['ItemCode'].nunique(),
        'date_min': df['Date'].min().date(),
        'date_max': df['Date'].max().date(),
        'null_total': int(df.isna().sum().sum()),
        'duplicate_full_rows': int(df.duplicated().sum()),
    })

schema_audit = pd.DataFrame(schema_rows)
schema_audit


,dataset,rows,cols,sku_count,date_min,date_max,null_total,duplicate_full_rows
0,fixed_ffill,711980,7,15972,2020-11-17,2025-09-05,0,39443
1,fixed_median,711980,7,15972,2020-11-17,2025-09-05,0,38586
2,outlier_fixed_ffill,711980,7,15972,2020-11-17,2025-09-05,0,37379
3,outlier_fixed_median,711980,7,15972,2020-11-17,2025-09-05,0,37407


## 2. Price / Cost / Return Audit

Kiểm tra giá âm, cost âm, return rows và các trường hợp `Quantity > 0` nhưng giá/cost âm.


In [5]:
price_rows = []
for name, df in datasets.items():
    q = df['Quantity']
    up = df['UnitPrice']
    uc = df['Unit Cost']
    sa = df['SalesAmount']
    ca = df['Cost Amount']

    price_rows.append({
        'dataset': name,
        'rows': len(df),
        'return_rows_qty_neg': int((q < 0).sum()),
        'qty_zero_rows': int((q == 0).sum()),
        'unitprice_neg': int((up < 0).sum()),
        'unitcost_neg': int((uc < 0).sum()),
        'qty_pos_price_neg': int(((q > 0) & (up < 0)).sum()),
        'qty_pos_cost_neg': int(((q > 0) & (uc < 0)).sum()),
        'qty_neg_price_pos': int(((q < 0) & (up > 0)).sum()),
        'qty_neg_cost_pos': int(((q < 0) & (uc > 0)).sum()),
        'sales_sign_mismatch': int(((q > 0) & (sa < 0)).sum() + ((q < 0) & (sa > 0)).sum()),
        'cost_sign_mismatch': int(((q > 0) & (ca < 0)).sum() + ((q < 0) & (ca > 0)).sum()),
    })

price_audit = pd.DataFrame(price_rows)
price_audit


,dataset,rows,return_rows_qty_neg,qty_zero_rows,unitprice_neg,unitcost_neg,qty_pos_price_neg,qty_pos_cost_neg,qty_neg_price_pos,qty_neg_cost_pos,sales_sign_mismatch,cost_sign_mismatch
0,fixed_ffill,711980,37434,3164,39829,43958,2514,6526,0,9,2514,6534
1,fixed_median,711980,37434,3164,37315,37417,0,22,0,11,0,28
2,outlier_fixed_ffill,711980,37434,3164,39489,43950,2174,6526,0,7,5,29
3,outlier_fixed_median,711980,37434,3164,37315,37417,0,22,0,7,5,29


## 3. Sales / Cost Recalculation Error Audit

So sánh `SalesAmount` với `Quantity * UnitPrice`, và `Cost Amount` với `Quantity * Unit Cost`.


In [6]:
error_rows = []
quantiles = [0.5, 0.9, 0.95, 0.99, 0.999]

for name, df in datasets.items():
    q = df['Quantity'].astype('float64')
    up = df['UnitPrice'].astype('float64')
    sa = df['SalesAmount'].astype('float64')
    uc = df['Unit Cost'].astype('float64')
    ca = df['Cost Amount'].astype('float64')

    sales_err = (sa - q * up).abs()
    cost_err = (ca - q * uc).abs()

    row = {'dataset': name}
    for p in quantiles:
        row[f'sales_err_q{int(p*1000):03d}'] = float(sales_err.quantile(p))
        row[f'cost_err_q{int(p*1000):03d}'] = float(cost_err.quantile(p))
    row['sales_err_mean'] = float(sales_err.mean())
    row['cost_err_mean'] = float(cost_err.mean())
    row['sales_err_gt_1m'] = int((sales_err > 1_000_000).sum())
    row['cost_err_gt_1m'] = int((cost_err > 1_000_000).sum())
    error_rows.append(row)

sales_error_audit = pd.DataFrame(error_rows)
sales_error_audit


,dataset,sales_err_q500,cost_err_q500,sales_err_q900,cost_err_q900,sales_err_q950,cost_err_q950,sales_err_q990,cost_err_q990,sales_err_q999,cost_err_q999,sales_err_mean,cost_err_mean,sales_err_gt_1m,cost_err_gt_1m
0,fixed_ffill,0.0,0.25,125000.00,0.5,560000.0,130001.209570,3136000.0,2.200000e+06,1.164508e+07,7.112535e+06,148133.342922,78795.696322,24278,16231
1,fixed_median,0.0,0.25,125000.00,0.5,560000.0,130001.209570,3136000.0,2.200000e+06,1.164508e+07,7.112535e+06,148133.343414,78795.696322,24278,16231
2,outlier_fixed_ffill,0.0,0.25,167273.25,0.5,720000.0,352424.375000,3880000.0,2.434733e+06,1.859002e+07,7.808931e+06,208554.629724,95621.732523,28919,19334
3,outlier_fixed_median,0.0,0.25,160000.00,0.5,675000.0,241228.437695,3700000.0,2.214883e+06,1.824076e+07,7.187854e+06,197236.696184,83135.552589,27489,16595


## 4. Calendar Audit

Kiểm tra ngày thực tế có giao dịch, Chủ nhật, và last-28 calendar window.


In [7]:
def calendar_stats(df: pd.DataFrame) -> dict:
    dates = pd.Index(sorted(df['Date'].dt.normalize().unique()))
    date_min = dates.min()
    date_max = dates.max()
    full_dates = pd.date_range(date_min, date_max, freq='D')
    missing_dates = full_dates.difference(dates)
    sunday_full = full_dates[full_dates.dayofweek == 6]
    sunday_present = dates[dates.dayofweek == 6]
    last28_calendar = pd.date_range(date_max - pd.Timedelta(days=27), date_max, freq='D')
    last28_missing = last28_calendar.difference(dates)
    business_valid_28 = dates[-28:]

    return {
        'unique_dates': len(dates),
        'full_calendar_dates': len(full_dates),
        'missing_calendar_dates': len(missing_dates),
        'sundays_total_calendar': len(sunday_full),
        'sundays_present': len(sunday_present),
        'sundays_missing': len(sunday_full.difference(sunday_present)),
        'last28_calendar_present': int(last28_calendar.isin(dates).sum()),
        'last28_calendar_missing': len(last28_missing),
        'last28_missing_dates': ', '.join(str(d.date()) for d in last28_missing),
        'business_valid_start_28': business_valid_28[0].date(),
        'business_valid_end_28': business_valid_28[-1].date(),
    }

calendar_audit = pd.DataFrame([
    {'dataset': name, **calendar_stats(df)}
    for name, df in datasets.items()
])
calendar_audit


,dataset,unique_dates,full_calendar_dates,missing_calendar_dates,sundays_total_calendar,sundays_present,sundays_missing,last28_calendar_present,last28_calendar_missing,last28_missing_dates,business_valid_start_28,business_valid_end_28
0,fixed_ffill,1411,1754,343,250,4,246,22,6,"2025-08-10, 2025-08-17, 2025-08-24, 2025-08-31...",2025-08-02,2025-09-05
1,fixed_median,1411,1754,343,250,4,246,22,6,"2025-08-10, 2025-08-17, 2025-08-24, 2025-08-31...",2025-08-02,2025-09-05
2,outlier_fixed_ffill,1411,1754,343,250,4,246,22,6,"2025-08-10, 2025-08-17, 2025-08-24, 2025-08-31...",2025-08-02,2025-09-05
3,outlier_fixed_median,1411,1754,343,250,4,246,22,6,"2025-08-10, 2025-08-17, 2025-08-24, 2025-08-31...",2025-08-02,2025-09-05


## 5. Daily Aggregation Audit

Gộp theo `Date, ItemCode` để kiểm tra net negative demand và sparsity.


In [8]:
daily_data = {}
daily_rows = []

for name, df in datasets.items():
    daily = (
        df.groupby(['Date', 'ItemCode'], observed=True, as_index=False)
        .agg(
            Quantity=('Quantity', 'sum'),
            SalesAmount=('SalesAmount', 'sum'),
            CostAmount=('Cost Amount', 'sum'),
        )
    )
    daily['y_clip'] = daily['Quantity'].clip(lower=0)
    daily['profit'] = daily['SalesAmount'] - daily['CostAmount']
    daily_data[name] = daily

    sku_nonzero = daily.assign(nonzero=daily['Quantity'] != 0).groupby('ItemCode', observed=True)['nonzero'].sum()
    daily_rows.append({
        'dataset': name,
        'daily_rows': len(daily),
        'daily_negative_qty_rows': int((daily['Quantity'] < 0).sum()),
        'daily_zero_qty_rows': int((daily['Quantity'] == 0).sum()),
        'daily_positive_qty_rows': int((daily['Quantity'] > 0).sum()),
        'total_net_quantity': float(daily['Quantity'].sum()),
        'total_clipped_quantity': float(daily['y_clip'].sum()),
        'median_nonzero_days_per_sku': float(sku_nonzero.median()),
        'p25_nonzero_days_per_sku': float(sku_nonzero.quantile(0.25)),
        'p75_nonzero_days_per_sku': float(sku_nonzero.quantile(0.75)),
        'max_nonzero_days_per_sku': int(sku_nonzero.max()),
    })

sku_sparsity_audit = pd.DataFrame(daily_rows)
sku_sparsity_audit


,dataset,daily_rows,daily_negative_qty_rows,daily_zero_qty_rows,daily_positive_qty_rows,total_net_quantity,total_clipped_quantity,median_nonzero_days_per_sku,p25_nonzero_days_per_sku,p75_nonzero_days_per_sku,max_nonzero_days_per_sku
0,fixed_ffill,507050,20733,9891,476426,2447253.0,2493664.0,6.0,2.0,23.0,1061
1,fixed_median,507050,20733,9891,476426,2447253.0,2493664.0,6.0,2.0,23.0,1061
2,outlier_fixed_ffill,507050,20733,9891,476426,2447253.0,2493664.0,6.0,2.0,23.0,1061
3,outlier_fixed_median,507050,20733,9891,476426,2447253.0,2493664.0,6.0,2.0,23.0,1061


## 6. WRMSSE Weight / Profit Audit

Kiểm tra SKU có weight = 0 và mức tập trung weight ở top SKU.


In [9]:
weight_rows = []
top_weight_tables = {}

for name, daily in daily_data.items():
    profit = daily.groupby('ItemCode', observed=True)['profit'].sum()
    profit_pos = profit.clip(lower=0)
    total_profit_pos = profit_pos.sum()
    weights = profit_pos / total_profit_pos if total_profit_pos > 0 else profit_pos * 0

    top = weights.sort_values(ascending=False).head(20).reset_index()
    top.columns = ['ItemCode', 'weight']
    top['cum_weight'] = top['weight'].cumsum()
    top_weight_tables[name] = top

    weight_rows.append({
        'dataset': name,
        'sku_count': len(profit),
        'sku_weight_zero': int((profit_pos <= 0).sum()),
        'sku_weight_positive': int((profit_pos > 0).sum()),
        'total_profit_raw': float(profit.sum()),
        'total_profit_positive': float(total_profit_pos),
        'top10_weight_share': float(weights.sort_values(ascending=False).head(10).sum()),
        'top50_weight_share': float(weights.sort_values(ascending=False).head(50).sum()),
        'top100_weight_share': float(weights.sort_values(ascending=False).head(100).sum()),
        'top500_weight_share': float(weights.sort_values(ascending=False).head(500).sum()),
    })

weight_audit = pd.DataFrame(weight_rows)
weight_audit


,dataset,sku_count,sku_weight_zero,sku_weight_positive,total_profit_raw,total_profit_positive,top10_weight_share,top50_weight_share,top100_weight_share,top500_weight_share
0,fixed_ffill,15972,1608,14364,1.579488e+11,1.649822e+11,0.119400,0.247779,0.337698,0.605364
1,fixed_median,15972,1372,14600,1.669957e+11,1.698359e+11,0.147232,0.272762,0.357743,0.612748
2,outlier_fixed_ffill,15972,1817,14155,1.690377e+11,1.721971e+11,0.210903,0.322471,0.398178,0.633820
3,outlier_fixed_median,15972,1817,14155,1.690377e+11,1.721971e+11,0.210903,0.322471,0.398178,0.633820


In [10]:
# Show top-weight SKUs for each dataset
for name, top in top_weight_tables.items():
    print('===', name, 'top 20 weights ===')
    display(top)


=== fixed_ffill top 20 weights ===


,ItemCode,weight,cum_weight
0,SKU-00003,0.048881,0.048881
1,SKU-00002,0.025307,0.074188
2,SKU-09760,0.007241,0.081429
3,SKU-00324,0.006010,0.087439
4,SKU-12537,0.005684,0.093122
5,SKU-14323,0.005654,0.098776
6,SKU-00005,0.005384,0.104160
7,SKU-14320,0.005374,0.109534
8,SKU-09647,0.005155,0.114689
9,SKU-12534,0.004711,0.119400


=== fixed_median top 20 weights ===


,ItemCode,weight,cum_weight
0,SKU-00003,0.046612,0.046612
1,SKU-09458,0.027109,0.073721
2,SKU-00002,0.024075,0.097797
3,SKU-08589,0.012510,0.110307
4,SKU-09760,0.007035,0.117341
5,SKU-12534,0.006963,0.124304
6,SKU-12537,0.006360,0.130664
7,SKU-00324,0.005852,0.136517
8,SKU-14323,0.005502,0.142018
9,SKU-14320,0.005213,0.147232


=== outlier_fixed_ffill top 20 weights ===


,ItemCode,weight,cum_weight
0,SKU-00003,0.096986,0.096986
1,SKU-00002,0.046532,0.143518
2,SKU-09458,0.015338,0.158856
3,SKU-00005,0.013028,0.171883
4,SKU-08589,0.007466,0.179350
5,SKU-12534,0.007061,0.186410
6,SKU-09760,0.006942,0.193352
7,SKU-12537,0.006428,0.199780
8,SKU-00324,0.005682,0.205461
9,SKU-14323,0.005442,0.210903


=== outlier_fixed_median top 20 weights ===


,ItemCode,weight,cum_weight
0,SKU-00003,0.096986,0.096986
1,SKU-00002,0.046532,0.143518
2,SKU-09458,0.015338,0.158856
3,SKU-00005,0.013028,0.171883
4,SKU-08589,0.007466,0.179350
5,SKU-12534,0.007061,0.186410
6,SKU-09760,0.006942,0.193352
7,SKU-12537,0.006428,0.199780
8,SKU-00324,0.005682,0.205461
9,SKU-14323,0.005442,0.210903


## 7. Cross-Dataset Summary


In [11]:
summary = (
    schema_audit
    .merge(price_audit, on='dataset')
    .merge(sales_error_audit, on='dataset')
    .merge(calendar_audit, on='dataset')
    .merge(sku_sparsity_audit, on='dataset')
    .merge(weight_audit, on='dataset')
)

selected_cols = [
    'dataset', 'rows', 'sku_count', 'unique_dates', 'missing_calendar_dates',
    'sundays_present', 'sundays_missing', 'last28_calendar_present', 'last28_calendar_missing',
    'return_rows_qty_neg', 'unitprice_neg', 'unitcost_neg', 'qty_pos_price_neg',
    'sales_err_q950', 'sales_err_q990', 'daily_negative_qty_rows',
    'median_nonzero_days_per_sku', 'sku_weight_zero', 'top100_weight_share'
]

summary[selected_cols]


KeyError: "['rows', 'sku_count'] not in index"

## 8. Optional Export

Chạy cell này nếu muốn lưu bảng audit ra CSV để gửi/so sánh bên ngoài.


In [ ]:
EXPORT = False
if EXPORT:
    summary.to_csv('eda1_summary.csv', index=False)
    schema_audit.to_csv('eda1_schema_audit.csv', index=False)
    price_audit.to_csv('eda1_price_audit.csv', index=False)
    sales_error_audit.to_csv('eda1_sales_error_audit.csv', index=False)
    calendar_audit.to_csv('eda1_calendar_audit.csv', index=False)
    sku_sparsity_audit.to_csv('eda1_sku_sparsity_audit.csv', index=False)
    weight_audit.to_csv('eda1_weight_audit.csv', index=False)
    print('Exported EDA audit CSV files.')
else:
    print('EXPORT=False, no files written.')
